In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# ミッション1

下記の命令を組み合わせてプログラムを書き、ロボットを見えている標識(標識名はsign1)のある場所に向かって進むようにしよう。（標識が見えなくなったら停止しよう）

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|rotate|一定速度で回転する|w=回転速度[度/秒]|rotate(w=90)|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = Search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに最大20cmほどずれる
- スタート時の向きはランダムに最大1度ほどずれる

In [ ]:
class Mission1(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((4.0, 0.6), 0.2, should_stop=True),
        ]
        self.initial_xy = (1.7, 0.0)
        self.random_d_xy = (0.1, 0.2)
        self.random_d_yaw_deg = 1
        self.set_signs(
            [
                Sign(x=2.7, y=-0.1, name="sign1"),
                Sign(x=3.5, y=0.1, name="sign1"),
                Sign(x=4.2, y=0.7, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission1()).show()

In [ ]:
class Mission1(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((4.0, 0.6), 0.2, should_stop=True),
        ]
        self.initial_xy = (1.7, 0.0)
        self.random_d_xy = (0.1, 0.2)
        self.random_d_yaw_deg = 1
        self.set_signs(
            [
                Sign(x=2.7, y=-0.1, name="sign1"),
                Sign(x=3.5, y=0.1, name="sign1"),
                Sign(x=4.2, y=0.7, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, rotate, search, **kwargs):

        while True:
            pos = search()
            if pos is None:
                move(v=0)
            else:
                # ####### ここから下にプログラムを書こう
                move(v=1)
                # ####### ここより上にプログラムを書こう
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Mission1())
sim.run()
SimDrawer(sim).show()

# ヒント
- 標識の方向に進むには、車を標識の方を向くように回転させることが必要
- 車を標識の方を向くようにするには・・・
  - もし標識が車より右にあるなら右に回転
  - もし標識が車より左にあるなら左に回転
  - もし標識が車のほぼ正面にあるなら、まっすぐ進む
- 上記を高速で繰り返してみよう